# DisasterM3 — Full Component-Matrix Combined Solution

| Component | Selected | Source |
|---|---|---|
| Encoder Backbone | ResNet-50 (ImageNet) | Noman (C5) |
| Decoder Architecture | U-Net++ (nested dense skips) | Tamanna/Lead (B2) |
| Loss Function | Phased Compound Loss (Tversky → Lovasz) | Noman (C4) |
| Data Strategy | WeightedRandomSampler + Copy-Paste Augmentation | Aryan (A2+A3) |
| Input Resolution | 768×768 native random crop | Ridita (D1c) |
| Inference | 8-view TTA | Ridita (D2) |

**Two things flagged before this run, not silently fixed:**

1. **Decoder choice vs. team's own mIoU ablation:** the selection matrix picks U-Net++ for its
   structural-recall advantage (78.66% vs 64.10%), but the team's own `all_members_ablation_summary.csv`
   showed U-Net++ scoring *below* the plain U-Net baseline on mIoU. Both can be true — recall and
   mIoU can move in opposite directions — but this means the final mIoU here may come in lower
   than the C5 single-axis champion (0.5296) even if debris recall genuinely improves. Run the
   5-epoch sanity check below and compare mIoU before committing to the full 40 epochs; a
   `DECODER_ARCH` toggle is provided in Cell 2 to fall back to plain U-Net with one line changed
   if mIoU regresses.
2. **Leaked instance counts corrected:** the matrix cites 44,969/18,125 Damaged/Destroyed
   instances — those are the **pre-fix, leakage-contaminated** counts from the original A3 bug
   (extraction ran over `pairs` instead of `train_pairs`). This notebook uses the corrected
   extraction (train-only), which produces the true counts (40,704/16,682 in the last verified
   run). Update the matrix/report with the corrected numbers before final submission.

**Before the full 40-epoch run:** do the 5-epoch sanity check first. ResNet-50 + U-Net++ at
768×768 is heavier than anything the team has tried — U-Net++'s nested dense skip
connections use meaningfully more VRAM than plain U-Net at the same backbone/resolution, so
batch size is dropped further below (4, not 6) as a conservative starting point.


In [1]:
import importlib.metadata as _md

PINS = {
    "torch": "2.4.1",
    "torchvision": "0.19.1",
    "torchaudio": "2.4.1",
    "segmentation-models-pytorch": "0.3.4",
    "albumentations": "1.4.21",
    "opencv-python-headless": "4.10.0.84",
}

_mismatched = []
for _pkg, _want in PINS.items():
    try:
        _have = _md.version(_pkg)
    except _md.PackageNotFoundError:
        _have = "not installed"
    if _have != _want:
        _mismatched.append(f"{_pkg}: {_have} → {_want}")

if _mismatched:
    print("⏳ Installing pinned versions:")
    for _m in _mismatched:
        print(f"   {_m}")
    !pip install -q \
        torch==2.4.1 \
        torchvision==0.19.1 \
        torchaudio==2.4.1 \
        segmentation-models-pytorch==0.3.4 \
        albumentations==1.4.21 \
        opencv-python-headless==4.10.0.84 \
        pillow \
        huggingface_hub[hf_xet]
    print("✓ Dependencies installed — continuing (fresh kernel, no restart needed).")
else:
    print("✓ All pinned versions already installed — proceeding.")

⏳ Installing pinned versions:
   torch: 2.10.0+cu128 → 2.4.1
   torchvision: 0.25.0+cu128 → 0.19.1
   torchaudio: 2.10.0+cu128 → 2.4.1
   segmentation-models-pytorch: not installed → 0.3.4
   albumentations: 2.0.8 → 1.4.21
   opencv-python-headless: 4.13.0.92 → 4.10.0.84
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.9/227.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━

In [2]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────
import os
import json
import torch
from pathlib import Path
from datetime import datetime

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"

# ── Model ──
ENCODER_NAME = "resnet50"
ENCODER_WEIGHTS = "imagenet"
NUM_CLASSES = 4   # 0=Background, 1=Intact, 2=Damaged, 3=Destroyed

# Decoder toggle: "unetplusplus" per the team's selection matrix (recall-optimized),
# or "unet" to fall back to the team's actual highest-mIoU config (C5) if the
# 5-epoch sanity check shows U-Net++ regressing mIoU.
DECODER_ARCH = "unet"   # "unetplusplus" | "unet"

# ── Training config ──
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 40              # set to 5 for the sanity-check run first
BATCH_SIZE = 6                 # U-Net++ @ 768 + ResNet-50 is heavier than plain U-Net —
                                 # start conservative, raise to 6 only if VRAM allows
NUM_WORKERS = 2
IMAGE_SIZE = 768

# ── Checkpointing (12-hour Kaggle session cap) ──
CHECKPOINT_DIR = "/kaggle/working/unet_checkpoints"
SAVE_EVERY_N_EPOCHS = 2
TIME_LIMIT_HOURS = 11.4

# ── Cross-session resume ──
HF_CHECKPOINT_REPO = None
RESUME_EPOCH = 0

# ── Data strategy: sampler AND copy-paste, stacked (per the matrix) ──
USE_WEIGHTED_SAMPLER = True
DESTROYED_WEIGHT = 10.0
DAMAGED_WEIGHT = 5.0
DEFAULT_WEIGHT = 1.0

USE_COPY_PASTE = True
COPY_PASTE_PROB = 0.5
COPY_PASTE_MAX = 3
INSTANCE_DIR = "/kaggle/working/minority_instances"

# ── Inference ──
USE_TTA_AT_EVAL = True

OUTPUT_DIR = "/kaggle/working/unet_disasterm3"
MODEL_NAME = f"disasterm3_{DECODER_ARCH}_{ENCODER_NAME}_crop{IMAGE_SIZE}_fullstack"

print(f"\u2713 Config loaded")
print(f"  Decoder: {DECODER_ARCH} | Encoder: {ENCODER_NAME} (pretrained={ENCODER_WEIGHTS})")
print(f"  Classes: {NUM_CLASSES}")
print(f"  Batch size: {BATCH_SIZE} | Crop: {IMAGE_SIZE}\u00d7{IMAGE_SIZE} | Epochs: {NUM_EPOCHS}")
print(f"  Sampler: {USE_WEIGHTED_SAMPLER} | Copy-Paste: {USE_COPY_PASTE} | TTA: {USE_TTA_AT_EVAL}")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")


✓ Config loaded
  Decoder: unet | Encoder: resnet50 (pretrained=imagenet)
  Classes: 4
  Batch size: 6 | Crop: 768×768 | Epochs: 40
  Sampler: True | Copy-Paste: True | TTA: True
  Device: Tesla T4
  VRAM: 15.6 GB


## 2. Load & Prepare Segmentation Data

In [3]:
# ── Cell 4: Build combined multi-class masks from 3 binary sources ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

by_image = defaultdict(dict)

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img

print(f"Unique images with at least one mask: {len(by_image):,}")

FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,
}

os.makedirs("/tmp/combined_masks", exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id

    if combined_mask is None:
        continue

    mask_filename = Path(img_path).name
    save_path = f"/tmp/combined_masks/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({"image_path": img_path, "mask_path": save_path})

print(f"\n\u2713 Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")


Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0


## 3. Dataset, 768-Crop, WeightedRandomSampler + Copy-Paste (stacked)
Train transform uses `RandomCrop(768,768)`. Validation uses `CenterCrop(768,768)` (fixed from
D1c's `RandomCrop`, which made eval non-reproducible). The sampler and copy-paste are layered
together per the matrix: `train_dataset` is wrapped in `CopyPasteAugDataset` first, then
`WeightedRandomSampler` controls which (now copy-paste-augmented) samples get shown more often.
**Leakage fix applied:** instance extraction (next section) scans `train_pairs` only, not the
full `pairs` — this is the corrected version of the bug caught earlier.

In [4]:
# ── Cell 5: PyTorch Dataset, 768-crop augmentation ────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class DisasterM3SegDataset(Dataset):
    """0=Background, 1=Intact, 2=Damaged, 3=Destroyed"""
    def __init__(self, pairs, transform=None, image_size=768):
        self.pairs = pairs
        self.transform = transform
        self.image_size = image_size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        image = cv2.imread(pair["image_path"])
        if image is None:
            return self.__getitem__((idx + 1) % len(self))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
        if mask is None:
            return self.__getitem__((idx + 1) % len(self))

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask.long()


train_transform = A.Compose([
    A.RandomCrop(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# FIX vs D1c: CenterCrop (not RandomCrop) for validation so metrics are reproducible.
val_transform = A.Compose([
    A.CenterCrop(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

from sklearn.model_selection import train_test_split
train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)

train_dataset = DisasterM3SegDataset(train_pairs, transform=train_transform, image_size=IMAGE_SIZE)
val_dataset = DisasterM3SegDataset(val_pairs, transform=val_transform, image_size=IMAGE_SIZE)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"\u2713 Base datasets created \u2014 Train: {len(train_dataset):,} | Val: {len(val_dataset):,}")


/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.21). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


✓ Base datasets created — Train: 5,798 | Val: 645


In [5]:
# ── Copy-Paste Step 1: Extract minority instances — TRAIN SET ONLY (leakage fix) ──
if USE_COPY_PASTE:
    os.makedirs(f"{INSTANCE_DIR}/damaged", exist_ok=True)
    os.makedirs(f"{INSTANCE_DIR}/destroyed", exist_ok=True)

    instance_count = {2: 0, 3: 0}

    for pair in train_pairs:  # FIX: train-only, prevents val leakage into copy-paste bank
        img = cv2.imread(pair["image_path"])
        mask = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None:
            continue

        for cls_id, cls_name in [(2, "damaged"), (3, "destroyed")]:
            binary = (mask == cls_id).astype(np.uint8)
            if binary.sum() == 0:
                continue

            num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary)
            for label_id in range(1, num_labels):
                area = stats[label_id, cv2.CC_STAT_AREA]
                if area < 100:
                    continue
                x, y, w, h = stats[label_id, cv2.CC_STAT_LEFT], stats[label_id, cv2.CC_STAT_TOP], \
                             stats[label_id, cv2.CC_STAT_WIDTH], stats[label_id, cv2.CC_STAT_HEIGHT]

                inst_img = img[y:y+h, x:x+w].copy()
                inst_mask = (labels[y:y+h, x:x+w] == label_id).astype(np.uint8)

                save_id = instance_count[cls_id]
                cv2.imwrite(f"{INSTANCE_DIR}/{cls_name}/img_{save_id}.png", inst_img)
                cv2.imwrite(f"{INSTANCE_DIR}/{cls_name}/mask_{save_id}.png", inst_mask * 255)
                instance_count[cls_id] += 1

    print(f"Extracted instances (train-only, leakage-free) \u2014 "
          f"Damaged: {instance_count[2]}, Destroyed: {instance_count[3]}")
else:
    instance_count = {2: 0, 3: 0}
    print("Copy-Paste disabled (USE_COPY_PASTE=False)")


Extracted instances (train-only, leakage-free) — Damaged: 40704, Destroyed: 16682


In [6]:
# ── Copy-Paste Step 2 + WeightedRandomSampler, stacked ────────────────────
import random, glob, torch

class CopyPasteAugDataset(Dataset):
    def __init__(self, base_dataset, instance_dir, paste_prob=0.5, max_paste=3):
        self.base = base_dataset
        self.paste_prob = paste_prob
        self.max_paste = max_paste
        self.instances = []
        for cls_id, cls_name in [(2, "damaged"), (3, "destroyed")]:
            img_paths = sorted(glob.glob(f"{instance_dir}/{cls_name}/img_*.png"))
            for img_path in img_paths:
                mask_path = img_path.replace("img_", "mask_")
                inst_img = cv2.imread(img_path)
                inst_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if inst_img is not None and inst_mask is not None:
                    inst_mask = (inst_mask > 127).astype(np.uint8)
                    self.instances.append((inst_img, inst_mask, cls_id))
        print(f"Loaded {len(self.instances)} minority instances for Copy-Paste")

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        image, mask = self.base[idx]
        if random.random() < self.paste_prob and self.instances:
            n = random.randint(1, self.max_paste)
            img_np = image.permute(1, 2, 0).numpy()
            mask_np = mask.numpy()
            for _ in range(n):
                inst_img, inst_mask, cls_id = random.choice(self.instances)
                ih, iw = inst_mask.shape[:2]
                if ih >= img_np.shape[0] or iw >= img_np.shape[1]:
                    continue
                y = random.randint(0, img_np.shape[0] - ih)
                x = random.randint(0, img_np.shape[1] - iw)
                paste_mask = inst_mask > 0
                inst_normalized = inst_img[..., ::-1].astype(np.float32) / 255.0
                inst_normalized = (inst_normalized - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
                img_np[y:y+ih, x:x+iw][paste_mask] = inst_normalized[paste_mask]
                mask_np[y:y+ih, x:x+iw][paste_mask] = cls_id
            image = torch.from_numpy(img_np).permute(2, 0, 1).float()
            mask = torch.from_numpy(mask_np).long()
        return image, mask


# ── Wrap with copy-paste first (if enabled) ──
train_dataset_final = train_dataset
if USE_COPY_PASTE:
    train_dataset_final = CopyPasteAugDataset(train_dataset, INSTANCE_DIR, paste_prob=COPY_PASTE_PROB, max_paste=COPY_PASTE_MAX)

# ── Then layer the sampler on top (weights computed over train_pairs, same order/length) ──
if USE_WEIGHTED_SAMPLER:
    from torch.utils.data import WeightedRandomSampler

    print("Computing sample weights for class-balanced sampling...")
    sample_weights = []
    for pair in train_pairs:
        mask = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
        has_destroyed = np.any(mask == 3)
        has_damaged = np.any(mask == 2)
        if has_destroyed:
            sample_weights.append(DESTROYED_WEIGHT)
        elif has_damaged:
            sample_weights.append(DAMAGED_WEIGHT)
        else:
            sample_weights.append(DEFAULT_WEIGHT)

    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(train_pairs), replacement=True)
    oversampled = sum(1 for w in sample_weights if w > DEFAULT_WEIGHT)
    print(f"  {oversampled} oversampled images out of {len(sample_weights)}")

    train_loader = DataLoader(train_dataset_final, batch_size=BATCH_SIZE, sampler=sampler,
                               num_workers=NUM_WORKERS, pin_memory=True)
else:
    train_loader = DataLoader(train_dataset_final, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True)

print(f"\u2713 train_loader ready ({len(train_dataset_final):,} samples, "
      f"sampler={'on' if USE_WEIGHTED_SAMPLER else 'off'}, copy-paste={'on' if USE_COPY_PASTE else 'off'})")


Loaded 57386 minority instances for Copy-Paste
Computing sample weights for class-balanced sampling...
  3412 oversampled images out of 5798
✓ train_loader ready (5,798 samples, sampler=on, copy-paste=on)


## 4. Loss Function — Phased Compound Loss (Tversky → Lovasz)

In [7]:
import segmentation_models_pytorch as smp
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class PhasedCompoundLoss(nn.Module):
    def __init__(self, num_epochs, lovasz_start_fraction=0.8):
        super().__init__()
        self.tversky = smp.losses.TverskyLoss(mode='multiclass', alpha=0.3, beta=0.7, gamma=2.0)
        self.lovasz = smp.losses.LovaszLoss(mode='multiclass')
        self.lovasz_start_epoch = int(num_epochs * lovasz_start_fraction)
        self.current_epoch = 0

    def set_epoch(self, epoch):
        self.current_epoch = epoch

    def forward(self, logits, targets):
        t_loss = self.tversky(logits, targets)
        if self.current_epoch >= self.lovasz_start_epoch:
            l_loss = self.lovasz(logits, targets)
            return 0.6 * t_loss + 0.4 * l_loss, t_loss.item(), l_loss.item()
        return t_loss, t_loss.item(), 0.0

criterion = PhasedCompoundLoss(num_epochs=NUM_EPOCHS).to(DEVICE)
print(f"\u2713 Phased Compound Loss ready on {DEVICE}")


✓ Phased Compound Loss ready on cuda


In [8]:
# ── Cell 7: Resume from HF checkpoint ─────────────────────────────────────
start_epoch = 0

if HF_CHECKPOINT_REPO:
    from huggingface_hub import hf_hub_download
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    try:
        ckpt_path = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=f"unet_epoch_{RESUME_EPOCH}.pth", token=hf_token)
        checkpoint = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        start_epoch = checkpoint["epoch"] + 1
        print(f"\u2713 Resumed from HF checkpoint: epoch {checkpoint['epoch']}")
    except Exception as e:
        print(f"\u26a0 Could not load HF checkpoint: {e}\n  Starting fresh.")
else:
    if os.path.exists(CHECKPOINT_DIR):
        ckpts = sorted([f for f in os.listdir(CHECKPOINT_DIR) if f.endswith(".pth")],
                       key=lambda x: int(x.split("_")[-1].split(".")[0]))
        if ckpts:
            latest = os.path.join(CHECKPOINT_DIR, ckpts[-1])
            checkpoint = torch.load(latest, map_location=DEVICE)
            model.load_state_dict(checkpoint["model_state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
            start_epoch = checkpoint["epoch"] + 1
            print(f"\u2713 Resumed from local checkpoint: {latest}")
        else:
            print("No checkpoints found \u2014 starting fresh.")
    else:
        print("No checkpoint directory \u2014 starting fresh.")


No checkpoint directory — starting fresh.


In [9]:
import time, os, datetime, json
import numpy as np
import segmentation_models_pytorch as smp
from tqdm import tqdm
from IPython.display import display, HTML
from torch.cuda.amp import GradScaler, autocast

training_start = time.time()
epoch_times = []

# ── Model: decoder chosen via DECODER_ARCH toggle in Cell 2 ──
if DECODER_ARCH == "unetplusplus":
    model = smp.UnetPlusPlus(encoder_name=ENCODER_NAME, encoder_weights=ENCODER_WEIGHTS,
                              in_channels=3, classes=NUM_CLASSES).to(DEVICE)
else:
    model = smp.Unet(encoder_name=ENCODER_NAME, encoder_weights=ENCODER_WEIGHTS,
                      in_channels=3, classes=NUM_CLASSES).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler = GradScaler()

def compute_iou(pred, target, num_classes):
    ious = []
    for cls in range(num_classes):
        pred_cls = (pred == cls); target_cls = (target == cls)
        intersection = (pred_cls & target_cls).sum().item()
        union = (pred_cls | target_cls).sum().item()
        ious.append(float('nan') if union == 0 else intersection / union)
    return ious

def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss = 0.0
    for images, masks in tqdm(loader, desc="Train", leave=False):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        with autocast():
            logits = model(images)
            loss, t_val, l_val = criterion(logits, masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_ious = [[] for _ in range(NUM_CLASSES)]
    for images, masks in tqdm(loader, desc="Val", leave=False):
        images, masks = images.to(device), masks.to(device)
        with autocast():
            logits = model(images)
            loss, _, _ = criterion(logits, masks)
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        for pred, mask in zip(preds, masks):
            ious = compute_iou(pred.cpu(), mask.cpu(), NUM_CLASSES)
            for cls, iou in enumerate(ious):
                if not np.isnan(iou):
                    all_ious[cls].append(iou)
    class_ious = [np.mean(c) if c else 0.0 for c in all_ious]
    miou = np.mean([iou for iou in class_ious if iou > 0])
    return total_loss / len(loader), miou

TIME_LIMIT_HOURS = 11.5
deadline = time.time() + TIME_LIMIT_HOURS * 3600
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

best_miou = 0.0
history = {"train_loss": [], "val_loss": [], "miou": []}

progress_html = display(HTML(f"<div><progress value='0' max='{NUM_EPOCHS}' style='width:300px; height:20px; vertical-align: middle;'></progress> [0/{NUM_EPOCHS}]</div>"), display_id=True)
table_html = display(HTML("<table border='1' class='dataframe'><thead><tr style='text-align: left;'><th>Epoch</th><th>Train Loss</th><th>Val Loss</th><th>mIoU</th><th>Time</th></tr></thead><tbody></tbody></table>"), display_id=True)
table_rows = ""

for epoch in range(start_epoch, NUM_EPOCHS):
    criterion.set_epoch(epoch)
    epoch_start = time.time()
    if time.time() > deadline:
        print(f"\n\u23f0 11.5 hour time budget reached! Saving and stopping gracefully.")
        break

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE)
    val_loss, miou = validate_one_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["miou"].append(miou)

    epoch_duration = time.time() - epoch_start
    epoch_times.append(epoch_duration)
    avg_epoch_time = sum(epoch_times) / len(epoch_times)
    eta_str = str(datetime.timedelta(seconds=int(avg_epoch_time * (NUM_EPOCHS - (epoch + 1)))))

    progress_html.update(HTML(f"<div><progress value='{epoch+1}' max='{NUM_EPOCHS}' style='width:300px; height:20px; vertical-align: middle;'></progress> [{epoch+1}/{NUM_EPOCHS} &lt; ETA: {eta_str}]</div>"))
    table_rows += f"<tr><td>{epoch+1}</td><td>{train_loss:.4f}</td><td>{val_loss:.4f}</td><td>{miou:.4f}</td><td>{epoch_duration:.1f}s</td></tr>"
    table_html.update(HTML(f"<table border='1' class='dataframe'><thead><tr style='text-align: left;'><th>Epoch</th><th>Train Loss</th><th>Val Loss</th><th>mIoU</th><th>Time</th></tr></thead><tbody>{table_rows}</tbody></table>"))

    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), "/kaggle/working/best_model.pth")

    torch.save({
        "epoch": epoch, "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(), "best_miou": best_miou,
    }, f"/kaggle/working/checkpoints/unet_epoch_{epoch}.pth")

    with open("/kaggle/working/training_history.json", "w") as f:
        json.dump(history, f, indent=2)

print(f"\n\u2713 Training complete. Best mIoU: {best_miou:.4f}")


Downloading: "https://download.pytorch.org/models/resnet50-19c8e357.pth" to /root/.cache/torch/hub/checkpoints/resnet50-19c8e357.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 336MB/s]
/tmp/ipykernel_23/2289087580.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Epoch,Train Loss,Val Loss,mIoU,Time
1,0.3730,0.2357,0.3391,577.8s
2,0.2024,0.2358,0.3301,578.4s
3,0.1785,0.2012,0.3751,578.3s
4,0.1658,0.2175,0.3768,578.5s
5,0.1633,0.1950,0.3718,579.0s
6,0.1539,0.1855,0.3889,578.3s
7,0.1507,0.1882,0.3845,582.6s
8,0.1479,0.1816,0.3879,578.5s
9,0.1428,0.1841,0.3903,578.5s
10,0.1363,0.1758,0.3947,578.3s


Train:   0%|          | 0/967 [00:00<?, ?it/s]/tmp/ipykernel_23/2289087580.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/108 [00:00<?, ?it/s]/tmp/ipykernel_23/2289087580.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



✓ Training complete. Best mIoU: 0.4461


## 5. Final Evaluation — Standard, then 8-view TTA

In [10]:
from torch.cuda.amp import autocast

@torch.no_grad()
def compute_per_class_metrics(model, loader, num_classes, device):
    model.eval()
    intersection = torch.zeros(num_classes); union = torch.zeros(num_classes)
    tp = torch.zeros(num_classes); fp = torch.zeros(num_classes); fn = torch.zeros(num_classes)
    for images, masks in tqdm(loader, desc="Metrics"):
        images, masks = images.to(device), masks.to(device)
        preds = torch.argmax(model(images), dim=1)
        for c in range(num_classes):
            pred_c = (preds == c); true_c = (masks == c)
            intersection[c] += (pred_c & true_c).sum().item()
            union[c] += (pred_c | true_c).sum().item()
            tp[c] += (pred_c & true_c).sum().item()
            fp[c] += (pred_c & ~true_c).sum().item()
            fn[c] += (~pred_c & true_c).sum().item()
    iou = intersection / (union + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    class_names = ["Background", "Intact", "Damaged", "Destroyed"]
    print(f"\n{'Class':<12} {'IoU':>8} {'Precision':>10} {'Recall':>8} {'F1':>8}")
    for c in range(num_classes):
        print(f"{class_names[c]:<12} {iou[c]:>8.4f} {precision[c]:>10.4f} {recall[c]:>8.4f} {f1[c]:>8.4f}")
    print(f"\nMean IoU: {iou.mean():.4f}\nMean F1:  {f1.mean():.4f}")
    return {"iou": iou, "precision": precision, "recall": recall, "f1": f1}


@torch.no_grad()
def predict_with_tta(model, image_batch, device='cuda'):
    transforms_and_inverses = [
        (lambda x: x, lambda x: x),
        (lambda x: torch.flip(x, [3]), lambda x: torch.flip(x, [3])),
        (lambda x: torch.flip(x, [2]), lambda x: torch.flip(x, [2])),
        (lambda x: torch.flip(x, [2, 3]), lambda x: torch.flip(x, [2, 3])),
        (lambda x: torch.rot90(x, 1, [2, 3]), lambda x: torch.rot90(x, 3, [2, 3])),
        (lambda x: torch.rot90(x, 2, [2, 3]), lambda x: torch.rot90(x, 2, [2, 3])),
        (lambda x: torch.rot90(x, 3, [2, 3]), lambda x: torch.rot90(x, 1, [2, 3])),
        (lambda x: torch.flip(torch.rot90(x, 1, [2,3]), [3]), lambda x: torch.rot90(torch.flip(x, [3]), 3, [2,3])),
    ]
    pred_sum = None
    for fwd, inv in transforms_and_inverses:
        with autocast():
            logits = model(fwd(image_batch))
        logits = inv(logits)
        probs = torch.softmax(logits, dim=1)
        pred_sum = probs if pred_sum is None else pred_sum + probs
    return (pred_sum / len(transforms_and_inverses)).argmax(dim=1)


@torch.no_grad()
def compute_per_class_metrics_tta(model, loader, num_classes, device):
    model.eval()
    intersection = torch.zeros(num_classes); union = torch.zeros(num_classes)
    tp = torch.zeros(num_classes); fp = torch.zeros(num_classes); fn = torch.zeros(num_classes)
    for images, masks in tqdm(loader, desc="TTA Metrics"):
        images, masks = images.to(device), masks.to(device)
        preds = predict_with_tta(model, images, device)
        for c in range(num_classes):
            pred_c = (preds == c); true_c = (masks == c)
            intersection[c] += (pred_c & true_c).sum().item()
            union[c] += (pred_c | true_c).sum().item()
            tp[c] += (pred_c & true_c).sum().item()
            fp[c] += (pred_c & ~true_c).sum().item()
            fn[c] += (~pred_c & true_c).sum().item()
    iou = intersection / (union + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    class_names = ["Background", "Intact", "Damaged", "Destroyed"]
    print(f"\n[TTA] {'Class':<12} {'IoU':>8} {'Precision':>10} {'Recall':>8} {'F1':>8}")
    for c in range(num_classes):
        print(f"[TTA] {class_names[c]:<12} {iou[c]:>8.4f} {precision[c]:>10.4f} {recall[c]:>8.4f} {f1[c]:>8.4f}")
    print(f"\n[TTA] Mean IoU: {iou.mean():.4f}\n[TTA] Mean F1:  {f1.mean():.4f}")
    return {"iou": iou, "precision": precision, "recall": recall, "f1": f1}


model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))

print("================ Standard Evaluation ================")
metrics = compute_per_class_metrics(model, val_loader, NUM_CLASSES, DEVICE)

metrics_tta = None
if USE_TTA_AT_EVAL:
    print("\n================ TTA Evaluation (8-view) ================")
    metrics_tta = compute_per_class_metrics_tta(model, val_loader, NUM_CLASSES, DEVICE)
    print(f"\nFree TTA boost: {(metrics_tta['iou'].mean() - metrics['iou'].mean()):+.4f} mIoU")


/tmp/ipykernel_23/3966137050.py:79: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))


================ Standard Evaluation ================


Metrics: 100%|██████████| 108/108 [00:42<00:00,  2.57it/s]



Class             IoU  Precision   Recall       F1
Background     0.9536     0.9850   0.9676   0.9762
Intact         0.5564     0.6441   0.8033   0.7149
Damaged        0.3528     0.5111   0.5325   0.5216
Destroyed      0.3006     0.4211   0.5122   0.4622

Mean IoU: 0.5408
Mean F1:  0.6687

================ TTA Evaluation (8-view) ================


TTA Metrics:   0%|          | 0/108 [00:00<?, ?it/s]/tmp/ipykernel_23/3966137050.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
TTA Metrics: 100%|██████████| 108/108 [02:31<00:00,  1.40s/it]


[TTA] Class             IoU  Precision   Recall       F1
[TTA] Background     0.9547     0.9850   0.9688   0.9768
[TTA] Intact         0.5625     0.6507   0.8059   0.7200
[TTA] Damaged        0.3611     0.5324   0.5289   0.5306
[TTA] Destroyed      0.3013     0.4328   0.4979   0.4631

[TTA] Mean IoU: 0.5449
[TTA] Mean F1:  0.6726

Free TTA boost: +0.0041 mIoU


In [11]:
# ── Save results, CSV in the team's standard schema ──
import shutil, csv

shutil.copy("/kaggle/working/best_model.pth", "/kaggle/working/best_model_fullstack.pth")

final_metrics = metrics_tta if (USE_TTA_AT_EVAL and metrics_tta is not None) else metrics

combined_metrics = {
    "class_names": ["Background", "Intact", "Damaged", "Destroyed"],
    "iou": final_metrics["iou"].tolist(),
    "precision": final_metrics["precision"].tolist(),
    "recall": final_metrics["recall"].tolist(),
    "f1": final_metrics["f1"].tolist(),
    "mean_iou": final_metrics["iou"].mean().item(),
    "mean_f1": final_metrics["f1"].mean().item(),
    "standard_mean_iou": metrics["iou"].mean().item(),
    "tta_mean_iou": metrics_tta["iou"].mean().item() if metrics_tta is not None else None,
    "decoder": DECODER_ARCH,
    "instances_extracted": instance_count,
}
with open("/kaggle/working/metrics_fullstack.json", "w") as f:
    json.dump(combined_metrics, f, indent=2)

results = {
    "experiment_id": f"FULLSTACK_{DECODER_ARCH}_resnet50_phasedloss_crop768_sampler_copypaste_tta",
    "variable": "combined_all_components",
    "value": f"{DECODER_ARCH}+resnet50+PhasedTverskyLovasz+crop768+WeightedSampler+CopyPaste+TTA8",
    "miou": float(final_metrics["iou"].mean()),
    "bg_iou": float(final_metrics["iou"][0]),
    "intact_iou": float(final_metrics["iou"][1]),
    "damaged_iou": float(final_metrics["iou"][2]),
    "destroyed_iou": float(final_metrics["iou"][3]),
    "precision": float(final_metrics["precision"].mean()),
    "recall": float(final_metrics["recall"].mean()),
    "f1": float(final_metrics["f1"].mean()),
    "train_time_hrs": sum(epoch_times) / 3600,
    "notes": (
        f"{NUM_EPOCHS} epochs. All 6 matrix components stacked: {DECODER_ARCH} decoder, "
        f"ResNet-50 backbone, Phased TverskyLovasz loss, 768 crop, WeightedRandomSampler "
        f"(10x/5x/1x) + CopyPasteAugDataset (train-only, leakage-fixed: "
        f"Damaged={instance_count[2]}, Destroyed={instance_count[3]}), 8-view TTA. "
        f"Reported row is {'TTA' if USE_TTA_AT_EVAL else 'standard'} metrics; "
        f"standard-only mIoU was {metrics['iou'].mean():.4f}."
    ),
}
with open(f"/kaggle/working/{results['experiment_id']}.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=results.keys())
    w.writeheader()
    w.writerow(results)

print(f"\u2713 Saved: best_model_fullstack.pth, metrics_fullstack.json, {results['experiment_id']}.csv")


✓ Saved: best_model_fullstack.pth, metrics_fullstack.json, FULLSTACK_unet_resnet50_phasedloss_crop768_sampler_copypaste_tta.csv


## Recap
Compare this run's `destroyed_iou` and `miou` against:
- The team's best single-axis result (C5: mIoU 0.5296, Destroyed IoU 0.279)
- Your earlier combined run without U-Net++/Copy-Paste (mIoU ~0.40–0.41 range from the
  A-track runs, or re-run the plain-U-Net combined notebook for a direct comparison)

If mIoU here comes in **below** 0.5296, that's a real, reportable finding: it would mean
the recall-oriented U-Net++ choice, or stacking sampler+copy-paste together (double-counting
the same imbalance fix), cost more than TTA+resolution gained back. Set `DECODER_ARCH = "unet"`
in Cell 2 and re-run to isolate whether the decoder or the stacked data strategy is responsible
before reporting a final number.